In [0]:
    # ============================================
# BRONZE → SILVER PIPELINE (FINAL CLEAN VERSION)
# ============================================

from pyspark.sql.functions import col, sum

# --------------------------------------------
# 1. READ BRONZE DATA (PARQUET)
# --------------------------------------------
df = spark.read.format("parquet") \
    .load("/Volumes/ecom-medallion-pipeline/bronze/mobile_orders")

print("Initial Row Count:", df.count())

# --------------------------------------------
# 2. FIX COLUMN NAMES (IMPORTANT)
# --------------------------------------------
# Convert all columns to lowercase + replace spaces with _
df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])

print("Columns after rename:", df.columns)

# --------------------------------------------
# 3. DATA CHECK (SIMPLE)
# --------------------------------------------

# Show sample
df.show(5)

# Schema
df.printSchema()

# NULL check
print("NULL VALUES:")
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()   

# Duplicate check
print("Total rows:", df.count())
print("After removing duplicates:", df.dropDuplicates().count())

# --------------------------------------------
# 4. DATA CLEANING
# --------------------------------------------

# Remove NULL important values
df = df.dropna(subset=["name", "selling_price", "mrp"])

# Remove duplicates
df = df.dropDuplicates()

# Fix data types
df = df.withColumn("selling_price", col("selling_price").cast("int")) \
       .withColumn("mrp", col("mrp").cast("int")) \
       .withColumn("ratings", col("ratings").cast("double"))

# Remove invalid values
df = df.filter(col("selling_price") > 0) \
       .filter(col("mrp") > 0) \
       .filter(col("selling_price") <= col("mrp"))

# --------------------------------------------
# 5. FINAL CHECK
# --------------------------------------------
print("Final Row Count:", df.count())
df.show(5)

# --------------------------------------------
# 6. WRITE TO SILVER (DELTA)
# --------------------------------------------
silver_path = "/Volumes/ecom-medallion-pipeline/silver/mobile_orders"

df.write.format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("✅ Silver Layer Created Successfully")

# ============================================
# END
# ============================================

Initial Row Count: 504
Columns after rename: ['name', 'brand', 'selling_price', 'mrp', 'discount', 'ratings', 'no_of_ratings', 'details', 'ingestion_time', 'batch_id', 'source_file', 'load_date']
+--------------------+-------+-------------+-----+--------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+
|                name|  brand|selling_price|  mrp|discount|ratings|       no_of_ratings|             details|      ingestion_time|            batch_id|         source_file| load_date|
+--------------------+-------+-------------+-----+--------+-------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+
|SAMSUNG Galaxy F1...|SAMSUNG|        11999|14999| 20% off|    4.4|101958 Ratings�&�...|['4 GB RAM | 64 G...|2026-05-02 23:10:...|12365694-4097-48a...|dbfs:/Volumes/eco...|2026-05-02|
|REDMI 9i Sport (C...|  REDMI|         7099| 9999| 29% off|    4.3|1